# Part IV walkthrough — the inverse problem

Parts I–III assumed the equation was known and the solution was wanted. Now invert
that. We have snapshots $\{u(t_k)\}$ from an experiment, a DNS, or a row of sensors,
and we want the operator, the closure, or sometimes the equation.

Keep two tasks apart, because methods that are strong at one are often weak at the
other:

| task | goal | interpretable model |
|---|---|---|
| prediction | forecast $u(t_{K+k})$ | not required |
| identification | recover the operator or the equations | the entire point |

Dynamic mode decomposition is aimed squarely at the second, and the tensor-network
version of it is the subject of this notebook.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_dmd as dmd
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 10})

## 1. A dataset with a known answer

A travelling wave, a slower travelling wave, and a decaying structure. Three spatial
structures, so three DMD modes, and we know exactly what the eigenvalues should be.

In [ ]:
n_x, n_t = 10, 9
N_x, N_t = 2 ** n_x, 2 ** n_t
x = np.linspace(0, 1, N_x, endpoint=False)
dt = 0.004
t = np.arange(N_t) * dt
c1, c2, decay = 1.0, -0.4, -0.3

X = (np.sin(2 * np.pi * (x[:, None] - c1 * t[None, :]))
     + 0.6 * np.cos(6 * np.pi * (x[:, None] - c2 * t[None, :]))
     + 0.5 * np.exp(decay * t)[None, :] * np.cos(10 * np.pi * x)[:, None])

expected = np.sort_complex(np.array([
    np.exp(2j * np.pi * c1 * dt), np.exp(-2j * np.pi * c1 * dt),
    np.exp(6j * np.pi * c2 * dt), np.exp(-6j * np.pi * c2 * dt),
    np.exp(decay * dt)]))

plt.imshow(X[:, ::8].T, aspect="auto", origin="lower", cmap="RdBu_r",
           extent=[0, 1, 0, t[-1]])
plt.xlabel("x"), plt.ylabel("t"), plt.title("the data")
plt.tight_layout()
print(f"snapshot matrix: {N_x:,d} x {N_t:,d} = {X.size:,d} numbers")

## 2. Standard DMD, and where its cost sits

$$X = [u_0 \cdots u_{T-2}], \quad X' = [u_1 \cdots u_{T-1}], \quad X = U\Sigma V^\dagger$$
$$\tilde A = U^\dagger X' V \Sigma^{-1}, \quad \tilde A = W\Lambda W^{-1}, \quad \Phi = UW$$

The eigenvalue gives a growth rate and a frequency; the eigenvector gives a spatial
mode. That is the appeal: the output is a set of individually interpretable structures,
not a black box.

The bottleneck is the first line. An SVD of an $N_x \times N_t$ matrix is prohibitive
long before the physics is.

In [ ]:
t0 = time.perf_counter()
ref = dmd.exact_dmd(X, rank=5)
t_exact = time.perf_counter() - t0
print(f"exact DMD: {t_exact:.3f} s")
print("eigenvalues :", np.sort_complex(ref["eigenvalues"]))
print("expected    :", expected)

## 3. Space and time in a single train

Here is the observation the whole of Part IV rests on. Binarise **both** $x$ and $t$ and
put them on one chain. Time stops being a loop and becomes more legs.

In [ ]:
cores = dmd.space_time_qtt(X, eps=1e-10)
print(f"sites        : {len(cores)}  ({n_x} space + {n_t} time)")
print(f"bond dims    : {tn.tt_ranks(cores)}")
print(f"parameters   : {tn.tt_size(cores):,d}  vs  {X.size:,d} dense "
      f"({X.size / tn.tt_size(cores):.0f}x)")
print(f"space/time bond: {tn.tt_ranks(cores)[n_x]}")

That last number is not a bookkeeping detail. **The bond at the space/time interface is
the temporal complexity of the dataset**, read off directly. Compare a travelling wave
with a standing wave:

In [ ]:
for name, data in [
    ("travelling  sin(2pi(x - ct))", np.sin(2 * np.pi * (x[:, None] - t[None, :]))),
    ("standing    sin(2pi x) cos(2pi t)",
     np.sin(2 * np.pi * x)[:, None] * np.cos(2 * np.pi * t)[None, :]),
    ("three modes (the data above)", X),
    ("white noise", np.random.default_rng(0).standard_normal((N_x, N_t))),
]:
    c = dmd.space_time_qtt(data, eps=1e-10)
    print(f"{name:>34}: space/time bond = {tn.tt_ranks(c)[n_x]:>4}")

A standing wave is a single product $f(x)g(t)$, so it is rank 1 across that cut.
A travelling wave is a sum of two, so rank 2. Noise has no structure to share and the
bond saturates. You have measured "how many degrees of freedom does this dataset have
in time" without running any algorithm.

## 4. MPS-DMD: the SVD is already there

Gauge the train to mixed canonical form at the space/time bond. The Schmidt
decomposition across that bond **is** $U\Sigma V^\dagger$ — the SVD that standard DMD
spends its whole budget computing. From there:

1. **gauge** — left block becomes an isometry $U$, the bond carries $\Sigma$;
2. **shift and project** — apply a rank-2 time-shift MPO and contract,
   $\tilde A = \Sigma (V^\dagger T V)\Sigma^{-1}$, a $\chi\times\chi$ matrix;
3. **eigendecompose** $\tilde A$. Modes are $\Phi = UW$, still in train form.

No snapshot matrix is formed at any point.

In [ ]:
t0 = time.perf_counter()
res = dmd.mps_dmd(cores, n_x, rank=5)
t_mps = time.perf_counter() - t0
print(f"MPS-DMD (given the train): {t_mps * 1e3:.1f} ms")
print("eigenvalues :", np.sort_complex(res["eigenvalues"]))
print("exact DMD   :", np.sort_complex(ref["eigenvalues"]))
print(f"\nmax difference: "
      f"{np.abs(np.sort_complex(res['eigenvalues']) - np.sort_complex(ref['eigenvalues'])).max():.2e}")

Identical to machine precision. Same modes, same eigenvalues, same predictions —
obtained from a $\chi\times\chi$ matrix instead of an $N_x\times N_t$ one.

### The one subtlety worth stating out loud

The non-periodic time shift sends the final snapshot to zero, so the raw projection is
$QP^\top$ rather than the least-squares $QP^{+}$ that DMD wants. Writing $v$ for the
last column of $V^\dagger$, $PP^\top = I - vv^\top$, so one Sherman–Morrison rank-one
correction recovers exact DMD. Skip it and every eigenvalue is damped by roughly
$1/N_t$ per step — invisible in the spectrum, ruinous after a few hundred steps.

In [ ]:
# What the uncorrected version would give:
g = tn.tt_canonicalise(cores, centre=n_x - 1)
c0 = g[n_x - 1]
r, d, r1 = c0.shape
u_, s_, vt_ = np.linalg.svd(c0.reshape(r * d, r1), full_matrices=False)
k = 5
u_, s_, vt_ = u_[:, :k], s_[:k], vt_[:k]
right = [np.tensordot(vt_, g[n_x], axes=([1], [0]))] + g[n_x + 1:]
shifted = tn.tt_round(tn.mpo_apply(tn.qtt_shift(n_t, -1), right), eps=1e-12)
M = dmd._overlap_matrix(shifted, right)
lam_raw = np.linalg.eigvals((s_[:, None] * M) / s_[None, :])
print("uncorrected |lambda|:", np.sort(np.abs(lam_raw)))
print("corrected   |lambda|:", np.sort(np.abs(res["eigenvalues"])))
print("true        |lambda|:", np.sort(np.abs(expected)))
print(f"\nuncorrected damping per step: {1 - np.abs(lam_raw).max():.2e}  "
      f"(compare 1/N_t = {1 / N_t:.2e})")

## 5. Cost against the length of the time series

Standard DMD is polynomial in $N_t$. MPS-DMD, *given the train*, is logarithmic: the
time part of the chain has $\log_2 N_t$ cores and the shift MPO has rank 2.

In [ ]:
print(f"{'N_t':>8} {'exact DMD (s)':>15} {'MPS-DMD (ms)':>14} {'chi':>5}")
for nt in (6, 8, 10, 12):
    Nt = 2 ** nt
    tt = np.arange(Nt) * dt
    data = (np.sin(2 * np.pi * (x[:, None] - c1 * tt[None, :]))
            + 0.6 * np.cos(6 * np.pi * (x[:, None] - c2 * tt[None, :]))
            + 0.5 * np.exp(decay * tt)[None, :] * np.cos(10 * np.pi * x)[:, None])
    c = dmd.space_time_qtt(data, eps=1e-10)

    t0 = time.perf_counter()
    dmd.exact_dmd(data, rank=5)
    te = time.perf_counter() - t0

    t0 = time.perf_counter()
    r_ = dmd.mps_dmd(c, n_x, rank=5)
    tm = time.perf_counter() - t0
    print(f"{Nt:>8,d} {te:>15.3f} {tm * 1e3:>14.1f} {r_['rank']:>5}")

Two honest caveats on that table.

* The MPS-DMD column excludes building the space-time train. That is the right
  comparison when the train came *out of* a space-time solver, which is the intended
  setting. For raw external data you stack snapshot trains one at a time and truncate
  after each — linear in the number of snapshots, and done once.
* The exact-DMD column is dominated by LAPACK, which is very good. The crossover is a
  matter of problem size, not of one method being cleverer.

## 6. Prediction

$u(T+k) = U W \Lambda^k W^{-1} U^\dagger u_T$: one diagonal multiply and one contraction
of the open bond. The cost does not depend on how far ahead you predict.

In [ ]:
future = [0, N_t // 2, N_t - 1, N_t + 500, N_t + 2000]
print(f"{'step':>8} {'relative error':>16}  {'':>4}")
for k in future:
    pred = dmd.predict_dense(res, k)
    truth = (np.sin(2 * np.pi * (x - c1 * k * dt))
             + 0.6 * np.cos(6 * np.pi * (x - c2 * k * dt))
             + 0.5 * np.exp(decay * k * dt) * np.cos(10 * np.pi * x))
    tag = "within data" if k < N_t else "extrapolated"
    print(f"{k:>8,d} {np.linalg.norm(pred - truth) / np.linalg.norm(truth):>16.2e}  {tag}")

plt.plot(x, dmd.predict_dense(res, N_t + 2000), label="MPS-DMD, 2000 steps ahead")
k = N_t + 2000
plt.plot(x, (np.sin(2 * np.pi * (x - c1 * k * dt))
             + 0.6 * np.cos(6 * np.pi * (x - c2 * k * dt))
             + 0.5 * np.exp(decay * k * dt) * np.cos(10 * np.pi * x)),
         "--", label="truth")
plt.legend(), plt.xlabel("x")
plt.tight_layout()

It extrapolates because it identified the *operator*, not because it memorised the
data. That is the difference between the two tasks in the table at the top, and it is
also the honest limit of the method: if the true dynamics are not close to linear in
the observables you gave it, no amount of data will rescue the extrapolation.

## 7. What noise does

Truncation looks like denoising. Sometimes it is. There is no theory saying when.

In [ ]:
rng = np.random.default_rng(5)
clean = X
print(f"{'noise level':>12} {'chi (eps=1e-2)':>16} {'eig error, no trunc':>22} "
      f"{'eig error, chi=5':>18}")
for sigma in (0.0, 0.01, 0.05, 0.2):
    noisy = clean + sigma * rng.standard_normal(clean.shape)
    c_loose = dmd.space_time_qtt(noisy, eps=1e-2)
    r_full = dmd.mps_dmd(dmd.space_time_qtt(noisy, eps=1e-10), n_x, rank=5)
    r_trunc = dmd.mps_dmd(dmd.space_time_qtt(noisy, eps=1e-10, chi_max=5), n_x, rank=5)

    def eig_err(r):
        got = np.sort_complex(r["eigenvalues"])
        return np.abs(got - expected).max()

    print(f"{sigma:>12.2f} {max(tn.tt_ranks(c_loose)):>16} "
          f"{eig_err(r_full):>22.2e} {eig_err(r_trunc):>18.2e}")

Truncating to the true rank helps: the noise lives in the discarded singular values.
But the cross-over is empirical, the "true rank" is exactly what you did not know, and
nothing in the output tells you whether you truncated the noise or the physics. This is
an open problem, not a solved one — as are uncertainty quantification on tensor-network
predictions and the extension of this analysis beyond one spatial dimension.

What tensor networks *do* have over a neural surrogate here is that the answer comes out
as modes, frequencies and growth rates. You can argue with it.